In [1]:
!ls processed_bert

X_bert.npy                X_nrc_features_scaled.npy
X_combined.npy            y.npy


In [2]:
data_dir = 'processed_bert'
X_combined_file = 'X_combined.npy'
y_file = 'y.npy'

In [3]:
import numpy as np
from os import path
import pandas as pd

X_combined_dataset = np.load(path.join(data_dir, X_combined_file))
y_resampled = np.load(path.join(data_dir, y_file))
y_resampled = pd.Series(y_resampled)

In [4]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X_combined_dataset, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

print(f'X_train shape: {X_train_val.shape}')
print(f'y_train shape: {y_train_val.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

print(f'y_train count:\n{y_train_val.value_counts()}')
print(f'y_test count:\n{y_test.value_counts()}')

X_train shape: (79672, 394)
y_train shape: (79672,)
X_test shape: (19918, 394)
y_test shape: (19918,)
y_train count:
0    46724
1    32948
Name: count, dtype: int64
y_test count:
0    11681
1     8237
Name: count, dtype: int64


In [5]:
import tensorflow as tf

# BERT embeddings are dense numpy arrays — no custom data generator needed.
# Keras accepts dense arrays directly in fit/evaluate/predict.
print(f"TensorFlow version: {tf.__version__}")
print(f"X_combined_dataset dtype: {X_combined_dataset.dtype}, shape: {X_combined_dataset.shape}")

TensorFlow version: 2.16.1
X_combined_dataset dtype: float64, shape: (99590, 394)


In [6]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

# Split first to avoid leakage from duplicated samples entering validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.2, stratify=y_train_val, random_state=42
)

ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

# Resample full train+val only for final retraining after model selection.
X_train_val_resampled, y_train_val_resampled = ros.fit_resample(X_train_val, y_train_val)

print(f'X_train_resampled shape: {X_train_resampled.shape}')
print(f'y_train_resampled shape: {y_train_resampled.shape}')
print(f'y_train_resampled count:\n{y_train_resampled.value_counts()}')

X_train_resampled shape: (74758, 394)
y_train_resampled shape: (74758,)
y_train_resampled count:
0    37379
1    37379
Name: count, dtype: int64


In [7]:
# Validation set remains untouched (no oversampling).
print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}')
print(f'y_train count:\n{y_train.value_counts()}')
print(f'y_val count:\n{y_val.value_counts()}')

X_train shape: (63737, 394), X_val shape: (15935, 394)
y_train count:
0    37379
1    26358
Name: count, dtype: int64
y_val count:
0    9345
1    6590
Name: count, dtype: int64


In [8]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    # Add batch normalization at input
    model.add(layers.BatchNormalization())

    # Increase depth and width options
    for i in range(hp.Int('num_layers', min_value=2, max_value=5)):
        model.add(
            layers.Dense(
                units=hp.Int(f'units_hidden_{i}', min_value=64, max_value=768, step=64),
                activation=hp.Choice(f'activation_hidden_{i}', values=['relu', 'tanh', 'elu'])
            )
        )
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(rate=hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)))

    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))

    # Tune optimizer
    optimizer_choice = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'])
    learning_rate = hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='log')

    if optimizer_choice == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_choice == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.AUC(curve='PR', name='pr_auc')
        ]
    )
    return model

In [9]:
from keras_tuner import Hyperband
import keras_tuner as kt

PROJECT_NAME = 'reddit_mlp_bert_nrclex_v4'
tuner = Hyperband(
    build_model,
    objective=kt.Objective('val_pr_auc', direction='max'),
    max_epochs=30,
    factor=3,
    hyperband_iterations=2,
    directory='keras_mlp',
    project_name=PROJECT_NAME,
    overwrite=False  # Keep False to resume from previous crashed run
 )

Reloading Tuner from keras_mlp/reddit_mlp_bert_nrclex_v4/tuner0.json


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Re-running this cell will continue pending tuner trials when overwrite=False.
tuner.search(
    X_train_resampled, y_train_resampled,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
 )

Trial 172 Complete [00h 32m 30s]
val_pr_auc: 0.8679184913635254

Best val_pr_auc So Far: 0.9188034534454346
Total elapsed time: 17h 55m 20s

Search: Running Trial #173

Value             |Best Value So Far |Hyperparameter
4                 |2                 |num_layers
384               |64                |units_hidden_0
relu              |elu               |activation_hidden_0
0.1               |0.3               |dropout_0
320               |64                |units_hidden_1
elu               |elu               |activation_hidden_1
0.2               |0.5               |dropout_1
rmsprop           |adam              |optimizer
0.0056895         |0.00027888        |learning_rate
320               |128               |units_hidden_2
tanh              |tanh              |activation_hidden_2
0.5               |0.1               |dropout_2
448               |576               |units_hidden_3
elu               |relu              |activation_hidden_3
0.5               |0.1               |dro

In [ ]:
# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
model = build_model(best_hps)

model.summary()

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Calculate class weights to combine with oversampling for better balance
from sklearn.utils import class_weight
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_resampled),
    y=y_train_resampled
)
class_weight_dict = dict(enumerate(weights))

history = model.fit(
    X_train_resampled, y_train_resampled,
    epochs=200, batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Tune decision threshold on validation set to optimize F1.
from sklearn.metrics import f1_score

val_probs = model.predict(X_val, batch_size=32).ravel()
threshold_grid = np.linspace(0.1, 0.9, 81)
f1_scores = [f1_score(y_val, (val_probs >= t).astype(int)) for t in threshold_grid]
best_threshold = float(threshold_grid[int(np.argmax(f1_scores))])

print(f'Best validation threshold (F1): {best_threshold:.3f}')
print(f'Best validation F1: {max(f1_scores):.4f}')

In [ ]:
best_epoch_num = np.argmin(history.history['val_loss']) + 1

print(f'Best epoch found: {best_epoch_num}')

final_model = build_model(best_hps)

# Calculate class weights for the full training set
weights_full = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_val_resampled),
    y=y_train_val_resampled
)
class_weight_dict_full = dict(enumerate(weights_full))

final_model.fit(X_train_val_resampled, y_train_val_resampled,
                epochs=best_epoch_num, batch_size=32, verbose=True,
                class_weight=class_weight_dict_full)

In [ ]:
loss, accuracy, auc, pr_auc = final_model.evaluate(X_test, y_test.values, batch_size=32)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {auc:.4f}")
print(f"Test PR-AUC: {pr_auc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

y_pred_probs = final_model.predict(X_test, batch_size=32).ravel()
y_pred = (y_pred_probs >= best_threshold).astype("int32")

print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Calculate PR curve and Average Precision
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, _ = precision_recall_curve(y_test, y_pred_probs)
average_precision = average_precision_score(y_test, y_pred_probs)

# Plot PR curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label='PR curve (area = %0.2f)' % average_precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

In [ ]:
import os
model_directory='models'
if not os.path.exists(model_directory):
    os.makedirs(model_directory)
model_name=f'{PROJECT_NAME}.keras'
final_model.save(path.join(model_directory, model_name))